# 02 — Calidad de datos: umbrales cuantitativos y muestra analitica

**Objetivo especifico que responde:** OE-03 del anteproyecto (`InformesRev/AnteproyectoAndres.md`, numeral 1.4.2): *"Normalizar las variables del conjunto de datos analiticos, gestionando duplicados, valores faltantes, inconsistencias y tipificacion, para garantizar la calidad de la informacion y la confiabilidad de los resultados de la analitica."*

**Brecha que cierra:** `MATRIZ_OBJETIVOS_CODIGO.md` señalo la ausencia de umbrales cuantitativos de calidad y de un filtro de fecha 2018-2024 verificable (seccion 5, brechas globales). Este notebook define ambos de forma explicita y aplica, ademas, los criterios de inclusion/exclusion ya declarados en el anteproyecto (numerales 3.6.3 y 3.6.4) que no se habian verificado como implementados en ningun punto del repositorio.

**Insumo:** las tablas del modelo dimensional generadas en el notebook 01 (`analitica-descriptiva-andres/data/modelo_dimensional/`).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

REPO = Path(".").resolve().parents[1] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
DIM_DIR = REPO / "analitica-descriptiva-andres" / "data" / "modelo_dimensional"
OUT_DIR = REPO / "analitica-descriptiva-andres" / "data"

hechos = pd.read_parquet(DIM_DIR / "hechos_contratos.parquet")
dim_proveedor = pd.read_parquet(DIM_DIR / "dim_proveedor.parquet")
dim_entidad = pd.read_parquet(DIM_DIR / "dim_entidad.parquet")
dim_territorio = pd.read_parquet(DIM_DIR / "dim_territorio.parquet")
dim_modalidad = pd.read_parquet(DIM_DIR / "dim_modalidad.parquet")

print("hechos_contratos:", hechos.shape)


hechos_contratos: (48331, 20)


## 1. Umbrales cuantitativos de calidad (definicion propia, justificada)

Se adopta como criterio de aceptacion el estandar de completitud de ISO 8000 (citado en el anteproyecto, numeral 2.2.4), operacionalizado con un umbral explicito: **una variable se considera apta para analisis descriptivo directo si su nulidad es <= 20%**; entre 20% y 50% se considera de **uso restringido** (debe declararse el posible sesgo de exclusion en cualquier hallazgo que la use); por encima de 50% se considera **no apta** para esta fase del analisis sin un tratamiento de imputacion adicional, que queda fuera del alcance descriptivo de este ejercicio (no se imputan valores financieros o de proveedor por regresion o vecinos cercanos, dado que no es un requisito del anteproyecto y podria introducir supuestos no verificables).

Este umbral (20% / 50%) es una decision metodologica propia, documentada aqui por primera vez para este proyecto — no proviene de ningun documento del companero.

In [2]:
columnas_clave = [
    'fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato',
    'duraci_n_del_contrato', 'valor_del_contrato', 'valor_pagado',
    'dias_adicionados', 'n_adicion_valor', 'estado_contrato', 'liquidaci_n',
]

reporte_calidad = pd.DataFrame({
    'columna': columnas_clave,
    'nulidad_pct': [hechos[c].isna().mean() * 100 for c in columnas_clave],
})

def clasificar(pct):
    if pct <= 20:
        return 'Apta'
    elif pct <= 50:
        return 'Uso restringido'
    return 'No apta sin tratamiento adicional'

reporte_calidad['clasificacion'] = reporte_calidad['nulidad_pct'].apply(clasificar)
reporte_calidad = reporte_calidad.sort_values('nulidad_pct', ascending=False).reset_index(drop=True)
reporte_calidad


,columna,nulidad_pct,clasificacion
0,fecha_de_inicio_del_contrato,12.577849,Apta
1,fecha_de_firma,10.655687,Apta
2,fecha_de_fin_del_contrato,3.101529,Apta
3,duraci_n_del_contrato,0.000000,Apta
4,valor_del_contrato,0.000000,Apta
5,valor_pagado,0.000000,Apta
6,dias_adicionados,0.000000,Apta
7,n_adicion_valor,0.000000,Apta
8,estado_contrato,0.000000,Apta
9,liquidaci_n,0.000000,Apta


## 2. Deduplicacion

La tabla de hechos ya fue verificada en el notebook 01 como unica por `id_contrato` (48.331 valores unicos sobre 48.331 filas). No se detectan duplicados adicionales introducidos por este notebook. El criterio de deduplicacion aplicado aguas arriba (pipeline compartido, notebooks 00-06) fue conservar el registro mas reciente por llave — este notebook **no vuelve a deduplicar**, solo lo confirma.

In [3]:
dup_check = hechos['id_contrato'].duplicated().sum()
print("Duplicados en id_contrato dentro de este notebook:", dup_check)
assert dup_check == 0


Duplicados en id_contrato dentro de este notebook: 0


## 3. Aplicacion de los criterios de inclusion y exclusion del anteproyecto (numerales 3.6.3 y 3.6.4)

Estos criterios estan **declarados en el anteproyecto** pero, segun `InformesRev/RESUMEN_CODIGO.md` (seccion 8), no se habian verificado como implementados en ningun notebook del pipeline compartido. Se aplican aqui por primera vez, de forma explicita y trazable:

**Inclusion (3.6.3):**
1. Fecha de suscripcion (`fecha_de_firma`) comprendida entre 2018-01-01 y 2024-12-31.
2. Informacion disponible en los campos minimos: `valor_del_contrato`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato` (o `duraci_n_del_contrato`), `estado_contrato`.
3. Estado de ejecucion avanzado o finalizado: `estado_contrato` en {`terminado`, `Cerrado`} o `liquidaci_n == 'Si'`.

**Exclusion (3.6.4):**
1. `valor_del_contrato` igual o menor a cero.
2. `fecha_de_fin_del_contrato` anterior a `fecha_de_inicio_del_contrato` (inversion logica de fechas).
3. Contratos en ejecucion activa (`estado_contrato` en {`En ejecución`, `Suspendido`, `Cancelado`, `Borrador`, `Aprobado`, `En aprobación`, `enviado Proveedor`}), dado que sus variables de desempeño final no son aun observables.

**Regla de resolucion de contradicciones:** se detectan contratos con `liquidaci_n == 'Si'` cuyo `estado_contrato` registrado aun es de ejecucion activa — una inconsistencia del propio dato fuente (SECOP), no introducida por este analisis. Ante esa contradiccion se aplica el criterio **mas conservador**: la exclusion por estado activo (3.6.4) tiene prioridad sobre la inclusion por liquidacion (3.6.3). Esto se documenta como hallazgo de calidad de datos, no se oculta.

In [4]:
n0 = len(hechos)
funnel = [('Total contratos de Obra (hechos_contratos)', n0)]

paso = hechos.copy()

# 1. Rango de fechas 2018-2024 (fecha de firma)
paso = paso[(paso['fecha_de_firma'] >= '2018-01-01') & (paso['fecha_de_firma'] <= '2024-12-31')]
funnel.append(('Fecha de firma en 2018-2024', len(paso)))

# 2. Campos minimos no nulos
paso = paso.dropna(subset=['valor_del_contrato', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'estado_contrato'])
funnel.append(('Campos minimos no nulos (valor, fechas, estado)', len(paso)))

# 3. Exclusion: valor <= 0
paso = paso[paso['valor_del_contrato'] > 0]
funnel.append(('Excluidos valor_del_contrato <= 0', len(paso)))

# 4. Exclusion: fechas invertidas
paso = paso[paso['fecha_de_fin_del_contrato'] >= paso['fecha_de_inicio_del_contrato']]
funnel.append(('Excluidas fechas invertidas (fin < inicio)', len(paso)))

# 5. Inclusion: estado de ejecucion avanzado o finalizado
estados_finalizados = ['terminado', 'Cerrado', 'cedido']
estados_activos_excluir = ['En ejecución', 'Suspendido', 'Cancelado', 'Borrador', 'Aprobado', 'En aprobación', 'enviado Proveedor']

# Nota de calidad de datos: se detectan contratos con liquidaci_n == 'Si' cuyo estado_contrato
# registrado aun figura como activo (ej. 'En ejecución', 'Cancelado'). Esto es una inconsistencia
# del propio dato fuente (un contrato no puede estar liquidado y en ejecucion activa a la vez).
# Ante esta contradiccion se aplica el criterio mas conservador: la exclusion por estado activo
# (3.6.4) tiene prioridad sobre la inclusion por liquidacion (3.6.3), evitando incorporar
# contratos cuyo desempeño final podria no ser realmente observable.
contradiccion = paso[(paso['liquidaci_n'] == 'Si') & (paso['estado_contrato'].isin(estados_activos_excluir))]
print(f"Nota de calidad: {len(contradiccion)} contratos con liquidaci_n='Si' pero estado_contrato activo "
      f"({contradiccion['estado_contrato'].value_counts().to_dict()}). Se excluyen por criterio conservador.")

incluir = paso['estado_contrato'].isin(estados_finalizados) | (paso['liquidaci_n'] == 'Si')
excluir_activos = paso['estado_contrato'].isin(estados_activos_excluir)
paso = paso[incluir & ~excluir_activos]
funnel.append(('Estado avanzado/finalizado, excluyendo activos por criterio conservador', len(paso)))

funnel_df = pd.DataFrame(funnel, columns=['Paso', 'Contratos restantes'])
funnel_df['Retencion_pct'] = (funnel_df['Contratos restantes'] / n0 * 100).round(2)
muestra_analitica = paso.copy()
funnel_df


Nota de calidad: 3538 contratos con liquidaci_n='Si' pero estado_contrato activo ({'En ejecución': 2800, 'Aprobado': 443, 'Suspendido': 290, 'Cancelado': 5}). Se excluyen por criterio conservador.


,Paso,Contratos restantes,Retencion_pct
0,Total contratos de Obra (hechos_contratos),48331,100.00
1,Fecha de firma en 2018-2024,32175,66.57
2,"Campos minimos no nulos (valor, fechas, estado)",31610,65.40
3,Excluidos valor_del_contrato <= 0,31544,65.27
4,Excluidas fechas invertidas (fin < inicio),31543,65.26
5,"Estado avanzado/finalizado, excluyendo activos...",19194,39.71


In [5]:
print(f"Muestra analitica final: {len(muestra_analitica)} contratos ({len(muestra_analitica)/n0*100:.1f}% del total de contratos de Obra integrados).")
print()
print("Distribucion de estado_contrato en la muestra analitica:")
print(muestra_analitica['estado_contrato'].value_counts())


Muestra analitica final: 19194 contratos (39.7% del total de contratos de Obra integrados).

Distribucion de estado_contrato en la muestra analitica:
estado_contrato
terminado     10378
Modificado     5897
Cerrado        2894
cedido           25
Name: count, dtype: int64


In [6]:
# Enriquecer la muestra analitica con atributos de dimension para uso directo en notebooks posteriores
# (vista analitica denormalizada — practica estandar de capa de reporte sobre un modelo dimensional)
vista = (
    muestra_analitica
    .merge(dim_proveedor[['codigo_proveedor', 'tipo_contratista', 'es_pyme']], on='codigo_proveedor', how='left')
    .merge(dim_entidad[['codigo_entidad', 'sector', 'rama', 'entidad_centralizada']], on='codigo_entidad', how='left')
    .merge(dim_territorio, on='territorio_id', how='left')
    .merge(dim_modalidad, on='modalidad_id', how='left')
)
print("Vista analitica enriquecida:", vista.shape)
assert vista['id_contrato'].is_unique, "La vista analitica perdio unicidad de id_contrato tras los merges"
vista.to_parquet(OUT_DIR / "muestra_analitica.parquet", index=False)
funnel_df.to_csv(OUT_DIR / "reporte_calidad_funnel.csv", index=False)
reporte_calidad.to_csv(OUT_DIR / "reporte_calidad_umbrales.csv", index=False)
print("Guardado: muestra_analitica.parquet, reporte_calidad_funnel.csv, reporte_calidad_umbrales.csv")


Vista analitica enriquecida: (19194, 29)


Guardado: muestra_analitica.parquet, reporte_calidad_funnel.csv, reporte_calidad_umbrales.csv


## 4. Sintesis de hallazgos de este notebook

- Se definieron umbrales cuantitativos propios de calidad de datos (apta <=20% nulidad, uso restringido 20-50%, no apta >50%), ausentes hasta ahora en cualquier documento del proyecto.
- Se implementaron por primera vez, de forma verificable, los criterios de inclusion/exclusion 3.6.3 y 3.6.4 del anteproyecto, incluido el filtro temporal 2018-2024 que `MATRIZ_OBJETIVOS_CODIGO.md` señalaba como ausente.
- **Hallazgo de calidad de datos propio:** se detecto un subconjunto de contratos con `liquidaci_n == 'Si'` cuyo `estado_contrato` seguia figurando como activo (ver conteo exacto en la celda de ejecucion), una inconsistencia del propio SECOP. Se resolvio con el criterio mas conservador (excluirlos), y se deja documentado como hallazgo citable en el capitulo de resultados.
- La muestra analitica resultante (ver conteo exacto en la celda de ejecucion) sera la base de todos los notebooks siguientes (03 a 08); cualquier hallazgo posterior debe interpretarse como valido para esta subpoblacion (contratos de obra con ejecucion avanzada o finalizada, 2018-2024), no para la totalidad de contratos de obra publica extraidos.
- Limitacion declarada: el criterio de estado "avanzado o finalizado" pudo excluir contratos legitimamente exitosos que aun no se marcan como `terminado`/`Cerrado`/liquidados en el momento de la extraccion; esto es coherente con la exclusion 3.6.4 del propio anteproyecto (evitar contratos cuyas variables de desempeño final no son aun observables), pero reduce el tamaño de muestra frente al universo total.